# Mathematical Engineering - Financial Engineering, FY 2025-2026

# Risk Management - Assignment 1: Hedging a Swaption Portfolio

**Case study:** The IR-derivative desk of Polimi Bank has the following positions opened today (15/02/2008):

- Long swaption payer 1m&10y 5y ATM - Notional €650 Mln
- Vanilla 10y IRS fixed rate receiver - Notional €550 Mln


In [47]:
# Importing the libraries
import numpy as np
import pandas as pd
import datetime as dt

from utilities.date_functions import (
    business_date_offset,
    date_series,
)
from utilities.ex0_utilities import bootstrap, plot_curve
from utilities.ex1_utilities import (
    swaption_price_calculator,
    irs_proxy_duration,
    swap_par_rate,
    swap_mtm,
    SwapType,
    shock_calculator,
)

The comment MODIFIED indicates that we changed part of code we found already written in the snippets

In [48]:
# =============================================================================
# Portfolio Parameters
# =============================================================================

# Swaption
swaption_maturity_y = 10  # Years component of swaption expiry
swaption_maturity_m = 1  # Months component of swaption expiry # MODIFIED
swaption_tenor_y = 5  # Underlying swap tenor in years
swaption_fixed_leg_freq = 1  # Annual fixed leg payments
swaption_type = SwapType.PAYER  
swaption_notional = 650_000_000
sigma_black = 0.7955  # Black implied volatility for the swaption

# IRS
irs_maturity = 10  # In years
irs_fixed_leg_freq = 1  # Annual fixed leg payments
irs_notional = 550_000_000
irs_type = SwapType.RECEIVER # MODIFIED

In [49]:
file_path = 'pkl/market_data_aggregated.p'

try:
    market_data = pd.read_pickle(file_path)
    print("Success.")
    
except FileNotFoundError:
    print(f"Error: File {file_path} not found.")


df_depos = market_data['depo']
df_futures = market_data['futures']
df_swaps = market_data['swaps']

Success.


In [50]:
# =============================================================================
# Bootstrap the discount curve from market data (Assignment 0)
# =============================================================================
today = dt.datetime(2008, 2, 15)
settlement_date = dt.datetime(2008, 2, 19)

discount_factors, zeroRates, dates = bootstrap(settlement_date, df_depos, df_futures, df_swaps, depo_idx=[0, 3], fut_idx=[0, 7])

## Q1: Mark-to-Market the portfolio at the mid-rate curve


In [51]:
# =============================================================================
# Q1: Compute the forward swap rate (= swaption strike, since ATM)
# =============================================================================

# Swaption expiry: today + 10y1m
swaption_expiry = business_date_offset(
    settlement_date, year_offset=swaption_maturity_y, month_offset=swaption_maturity_m 
)

# Underlying swap expiry: swaption expiry + 5y tenor
underlying_expiry = business_date_offset(
    settlement_date,
    year_offset=swaption_maturity_y + swaption_tenor_y,
    month_offset=swaption_maturity_m,
)

# Fixed leg schedule of the underlying forward-starting swap
swaption_underlying_fixed_leg_schedule = date_series(
    swaption_expiry, underlying_expiry, swaption_fixed_leg_freq
)

# Forward swap rate
fwd_swap_rate = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors,
    swaption_underlying_fixed_leg_schedule[0],
)
print(f"Forward swap rate: {fwd_swap_rate:.3%}")

Forward swap rate: 5.300%


In [52]:
# =============================================================================
# Q1: Portfolio MtM
# =============================================================================

strike = fwd_swap_rate # ATM

swaption_price, swaption_delta = swaption_price_calculator(
    fwd_swap_rate,
    strike,
    settlement_date, # MODIFIED
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors,
    swaption_type,
    compute_delta=True,
)


irs_expiry = business_date_offset(settlement_date, year_offset=irs_maturity) # MODIFIED
irs_fixed_leg_payment_dates = date_series(settlement_date, irs_expiry, irs_fixed_leg_freq)[1:]

irs_rate = swap_par_rate(irs_fixed_leg_payment_dates, discount_factors)
irs_mtm = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors, irs_type
)

# Portfolio MtM = swaption value + IRS value
ptf_mtm = swaption_notional * swaption_price + irs_notional * irs_mtm
print(f"Swaption price (per unit notional): \u20ac{swaption_price:.6f}")
print(f"IRS MtM (per unit notional):        \u20ac{irs_mtm:.6f}")
print(f"Portfolio MtM:                      \u20ac{ptf_mtm:,.2f}")


Swaption price (per unit notional): €0.115955
IRS MtM (per unit notional):        €-0.000000
Portfolio MtM:                      €75,370,456.41


## Q2: Evaluate the portfolio DV01-parallel


In [53]:
# =============================================================================
# Q2: Portfolio DV01-parallel (numerical)
# =============================================================================

discount_factors_up, _, _, = bootstrap(settlement_date, df_depos, df_futures, df_swaps, depo_idx=[0, 3], fut_idx=[0, 7], shock=0.0001) 

fwd_swap_rate_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under the shocked curve
swaption_price_up = swaption_price_calculator(
    fwd_swap_rate_up,
    strike,
    settlement_date, # MODIFIED
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_up,
    swaption_type,
)


irs_mtm_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_up, swap_type=irs_type
)

# Shocked portfolio MtM
ptf_mtm_up = swaption_notional * swaption_price_up + irs_notional * irs_mtm_up

# DV01-parallel
ptf_numeric_dv01 = ptf_mtm_up - ptf_mtm 
print(f"Portfolio DV01-parallel: \u20ac{ptf_numeric_dv01:,.2f}")


Portfolio DV01-parallel: €-367,999.97


## Q3: Analytical portfolio DV01 approximation


In [54]:
# =============================================================================
# Q3: Analytical portfolio DV01
# =============================================================================

irs_duration = irs_proxy_duration(
    settlement_date, irs_rate, irs_fixed_leg_payment_dates, discount_factors
)

ptf_proxy_dv01 = (
    swaption_notional * swaption_delta - irs_notional * irs_duration # MODIFICA IMPORTANTE 
) * 1e-4

print(f"Portfolio proxy DV01:    \u20ac{ptf_proxy_dv01:,.2f}")
print(f"Portfolio numeric DV01:  \u20ac{ptf_numeric_dv01:,.2f}")
print(f"Difference:              \u20ac{ptf_proxy_dv01 - ptf_numeric_dv01:,.2f}")

Portfolio proxy DV01:    €-294,118.50
Portfolio numeric DV01:  €-367,999.97
Difference:              €73,881.47


## Q4: Delta-hedge the portfolio with a 10y IRS


In [55]:
# =============================================================================
# Q4: Delta hedging with a 10y IRS
# =============================================================================

min_lot = 1_000_000  # IRS traded in multiples of €1M

irs_dv01 = irs_mtm_up - irs_mtm

delta_hedge_swap_notional = - ptf_numeric_dv01 / irs_dv01 

# Add irs_notional to obtain full position on IRS 10y
delta_hedge_swap_notional = irs_notional + round(delta_hedge_swap_notional / min_lot) * min_lot 


# Verify: hedged portfolio DV01 should be ~ 0
delta_hedge_dv01 = (
    swaption_notional * swaption_price_up + delta_hedge_swap_notional * irs_mtm_up
) - ptf_mtm
print(
    f"Numerical hedge: \u20ac{delta_hedge_swap_notional:,.0f} total IRS notional, residual DV01: \u20ac{delta_hedge_dv01:,.0f}"
)

# --- Analytical hedge (using the proxy DV01) ---
delta_hedge_swap_notional_proxy = round( ( swaption_notional * swaption_delta / irs_duration)/min_lot )*min_lot   
print(
    f"Analytical hedge: \u20ac{delta_hedge_swap_notional_proxy:,.0f} total IRS notional"
)
print(
    "\nThe analytical approximation significantly UNDERestimates the required hedge notional." # MODIFIED 
)

Numerical hedge: €91,000,000 total IRS notional, residual DV01: €31
Analytical hedge: €194,000,000 total IRS notional

The analytical approximation significantly UNDERestimates the required hedge notional.


In [56]:
# ALTERNATIVELY...

irs_dv01_proxy = - irs_duration*1e-4
notional_proxy = - ptf_proxy_dv01 / irs_dv01_proxy 

notional_proxy = irs_notional + round(notional_proxy / min_lot) * min_lot 

print(
    f"Hedge Proxy: \u20ac{notional_proxy:,.0f} total IRS notional"
)

Hedge Proxy: €194,000,000 total IRS notional


## Q5: Portfolio coarse-grained bucket DV01 (10y and 15y buckets)


In [57]:
# =============================================================================
# Q5: Coarse-Grained Bucket DV01 construction
# =============================================================================

macro_buckets = [dt.datetime(2018, 2, 19), dt.datetime(2023, 2, 19)] 

shock_list = shock_calculator(macro_buckets, df_depos, df_futures, df_swaps)

s10 = shock_list['10y']
s15 = shock_list['15y']

# !!! COMPLETE AS APPROPRIATE !!!
discount_factors_10y_up, _, _= bootstrap(settlement_date, df_depos, df_futures, df_swaps, depo_idx=[0, 3], fut_idx=[0, 7], shock=s10)
discount_factors_15y_up, _, _= bootstrap(settlement_date, df_depos, df_futures, df_swaps, depo_idx=[0, 3], fut_idx=[0, 7], shock=s15)

In [58]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 10y Bucket DV01
# =============================================================================

fwd_swap_rate_10y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_10y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

swaption_price_10y_up = swaption_price_calculator(
    fwd_swap_rate_10y_up,
    strike,
    settlement_date, # MODIFIED
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_10y_up,
    swaption_type,
    compute_delta=False,
)

# IRS MtM under 10y bucket shock
irs_mtm_10y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=irs_type
)

ptf_mtm_10y_up = (
    swaption_notional * swaption_price_10y_up + irs_notional * irs_mtm_10y_up
)

ptf_numeric_10y_dv01 = ptf_mtm_10y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 10y Bucket DV01: \u20ac{ptf_numeric_10y_dv01:,.2f}")

Portfolio Coarse-Grained 10y Bucket DV01: €-925,075.12


Portfolio: Long Receiver Swap + Long Payer Swaption.

View: Rates up (0-10y +1bp; 10-15y tapering).

Net Position: We are Net Short Rates. Since Swap Risk (8) > Swaption Risk (2), we lose if rates rise.

10-15y Bucket: We lose less here. The 10y Swap is over, so only the Swaption remains, which gains value as rates rise.


In [59]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 15y Bucket DV01
# =============================================================================

fwd_swap_rate_15y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_15y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under 15y bucket shock
swaption_price_15y_up = swaption_price_calculator(
    fwd_swap_rate_15y_up,
    strike,
    settlement_date, # MODIFIED
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_15y_up,
    swaption_type,
    compute_delta=False,
)

# IRS MtM under 15y bucket shock
irs_mtm_15y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_15y_up, irs_type
)

ptf_mtm_15y_up = (
    swaption_notional * swaption_price_15y_up
    + irs_notional * irs_mtm_15y_up
)

ptf_numeric_15y_dv01 = ptf_mtm_15y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 15y Bucket DV01: \u20ac{ptf_numeric_15y_dv01:,.2f}")

Portfolio Coarse-Grained 15y Bucket DV01: €557,584.75


The 15y bucket exposure is driven solely by the 1m & 10y-5y Payer Swaption. 

The 10y IRS has zero sensitivity in this bucket as it matures at year 10. 

Because the Payer Swaption underlying involves receiving the floating leg from years 10 to 15, a rate increase in this segment increases the swaption's value, resulting in a positive 15y bucket DV01.

## Q6: Delta-hedge with two liquid IRS (10y and 15y)


In [60]:
# =============================================================================
# Q6: Delta hedging with two IRS (10y and 15y)
# =============================================================================


import numpy.linalg as la

datex = dt.datetime(2018, 2, 19)
datey = dt.datetime(2023, 2, 20)


midprice_x = df_swaps.loc[datex, ['BID', 'ASK']].mean()
midprice_y = df_swaps.loc[datey, ['BID', 'ASK']].mean()

irs_fixed_leg_payment_dates_x = date_series(settlement_date, datex, irs_fixed_leg_freq)[1:]
irs_fixed_leg_payment_dates_y = date_series(settlement_date, datey, irs_fixed_leg_freq)[1:]

irs_mtm_x = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors, swap_type=SwapType.RECEIVER
)
irs_mtm_y = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors, swap_type=SwapType.RECEIVER
)


irs_mtm_x_10y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_10y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_x_15y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_15y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)


irs_x_10y_dv01 = irs_mtm_x_10y_up - irs_mtm_x
irs_y_10y_dv01 = irs_mtm_y_10y_up - irs_mtm_y

irs_x_15y_dv01 = irs_mtm_x_15y_up - irs_mtm_x
irs_y_15y_dv01 = irs_mtm_y_15y_up - irs_mtm_y

A = np.array([[irs_x_10y_dv01, irs_y_10y_dv01], [irs_x_15y_dv01, irs_y_15y_dv01]])
b = np.array([-ptf_numeric_10y_dv01, -ptf_numeric_15y_dv01])

delta_hedge = la.solve(A, b)


In [61]:

print(f"NEW position in IRS - 10y:   \u20ac{round(delta_hedge[0]/min_lot)*min_lot:,.2f}")

print(f"TOTAL position in IRS - 10y: \u20ac{round(delta_hedge[0]/min_lot)*min_lot + irs_notional:,.2f}")

print(f"TOTAL position in IRS - 15y: \u20ac{round(delta_hedge[1]/min_lot)*min_lot:,.2f}")


NEW position in IRS - 10y:   €-1,154,000,000.00
TOTAL position in IRS - 10y: €-604,000,000.00
TOTAL position in IRS - 15y: €517,000,000.00


## Q7: Curve flattening scenario


In [62]:
# =============================================================================
# Q7: Curve flattening scenario
# =============================================================================

diff_series = pd.Series(s10 - s15)

discount_factors_shock, zeroRates_shock, _ = bootstrap(settlement_date, df_depos, df_futures, df_swaps, depo_idx=[0, 3], fut_idx=[0, 7], shock=diff_series)


In [63]:
# a. PnL with portfolio hedged with 10y IRS (Receiver) - Notional 91 Mln Eur

# compute fwd swap rate with shocked curve 

fwd_swap_rate_shocked = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_shock,
    swaption_underlying_fixed_leg_schedule[0],
)

# new swaption price under shocked curve

swaption_price_shock = swaption_price_calculator(
    fwd_swap_rate_shocked, strike, settlement_date, swaption_expiry, underlying_expiry, sigma_black, swaption_fixed_leg_freq, discount_factors_shock, swaption_type
)


irs_mtm_shock_10y = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_shock, swap_type=irs_type
)


# ptf_mtm + ptf_mtm after shock

ptf_mtm_hedged = swaption_notional * swaption_price + delta_hedge_swap_notional * irs_mtm

ptf_mtm_after_shock = swaption_notional * swaption_price_shock + delta_hedge_swap_notional * irs_mtm_shock_10y


In [64]:
print(f"Hedged Portfolio:             \u20ac{ptf_mtm_hedged:,.2f}")
print(f"Hedged Portfolio After Shock: \u20ac{ptf_mtm_after_shock:,.2f}")
print(f"PnL (Hedge with 10y IRS):     \u20ac{ptf_mtm_after_shock - ptf_mtm_hedged:,.2f}")

Hedged Portfolio:             €75,370,456.41
Hedged Portfolio After Shock: €74,256,407.81
PnL (Hedge with 10y IRS):     €-1,114,048.60


In [65]:
# b.

# retrieve previously computed hedging notionals
notional_10y = round(delta_hedge[0]/min_lot) * min_lot
notional_15y = round(delta_hedge[1]/min_lot) * min_lot

# retrieve IRS Rate 15y - from df_swap
irs_15y = midprice_y

irs_15y_fixed_leg_payment_dates = date_series(settlement_date, datey, irs_fixed_leg_freq)[1:]

irs_mtm_shock_15y = swap_mtm(
    irs_15y, irs_15y_fixed_leg_payment_dates, discount_factors_shock, swap_type=SwapType.RECEIVER
)

ptf_mtm_hedged_2 = swaption_notional * swaption_price + notional_10y * irs_mtm + notional_15y * irs_mtm_y

ptf_mtm_after_shock_2 = swaption_notional * swaption_price_shock + notional_10y * irs_mtm_shock_10y + notional_15y * irs_mtm_shock_15y



In [66]:
print(f"Hedged Portfolio:                    \u20ac{ptf_mtm_hedged_2:,.2f}")
print(f"Hedged Portfolio After Shock:        \u20ac{ptf_mtm_after_shock_2:,.2f}")
print(f"PnL (Hedge with 2 IRS, 10y and 15y): \u20ac{ptf_mtm_after_shock_2 - ptf_mtm_hedged_2:,.2f}")

Hedged Portfolio:                    €75,370,456.41
Hedged Portfolio After Shock:        €75,812,412.24
PnL (Hedge with 2 IRS, 10y and 15y): €441,955.84


In [67]:
# SELECTION OF OTHER IRS - to be sure of our choice

datex = dt.datetime(2020, 2, 19)
datey = dt.datetime(2023, 2, 20)

midprice_x = df_swaps.loc[datex, ['BID', 'ASK']].mean()
midprice_y = df_swaps.loc[datey, ['BID', 'ASK']].mean()

irs_fixed_leg_payment_dates_x = date_series(today, datex, irs_fixed_leg_freq)[1:]
irs_fixed_leg_payment_dates_y = date_series(today, datey, irs_fixed_leg_freq)[1:]

irs_mtm_x = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors, swap_type=SwapType.RECEIVER
)
irs_mtm_y = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors, swap_type=SwapType.RECEIVER
)


irs_mtm_x_10y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_10y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_x_15y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_15y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)


irs_x_10y_dv01 = irs_mtm_x_10y_up - irs_mtm_x
irs_y_10y_dv01 = irs_mtm_y_10y_up - irs_mtm_y

irs_x_15y_dv01 = irs_mtm_x_15y_up - irs_mtm_x
irs_y_15y_dv01 = irs_mtm_y_15y_up - irs_mtm_y

A = np.array([[irs_x_10y_dv01, irs_y_10y_dv01], [irs_x_15y_dv01, irs_y_15y_dv01]])
b = np.array([-ptf_numeric_10y_dv01, -ptf_numeric_15y_dv01])

delta_hedge = la.solve(A, b)
print(f'The two positions are {round(delta_hedge[0]/min_lot)}M, {round(delta_hedge[1]/min_lot)}M')



The two positions are -1674M, 1088M


In [68]:

# SELECTION OF OTHER IRS - to be sure of our choice


datex = dt.datetime(2009, 2, 19)
datey = dt.datetime(2023, 2, 20)

midprice_x = df_swaps.loc[datex, ['BID', 'ASK']].mean()
midprice_y = df_swaps.loc[datey, ['BID', 'ASK']].mean()

irs_fixed_leg_payment_dates_x = date_series(today, datex, irs_fixed_leg_freq)[1:]
irs_fixed_leg_payment_dates_y = date_series(today, datey, irs_fixed_leg_freq)[1:]

irs_mtm_x = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors, swap_type=SwapType.RECEIVER
)
irs_mtm_y = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors, swap_type=SwapType.RECEIVER
)


irs_mtm_x_10y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_10y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_x_15y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_15y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)


irs_x_10y_dv01 = irs_mtm_x_10y_up - irs_mtm_x
irs_y_10y_dv01 = irs_mtm_y_10y_up - irs_mtm_y

irs_x_15y_dv01 = irs_mtm_x_15y_up - irs_mtm_x
irs_y_15y_dv01 = irs_mtm_y_15y_up - irs_mtm_y

A = np.array([[irs_x_10y_dv01, irs_y_10y_dv01], [irs_x_15y_dv01, irs_y_15y_dv01]])
b = np.array([-ptf_numeric_10y_dv01, -ptf_numeric_15y_dv01])

delta_hedge = la.solve(A, b)
print(f'The two positions are {round(delta_hedge[0]/min_lot)}M, {round(delta_hedge[1]/min_lot)}M')

The two positions are -9178M, 517M


In [69]:

# SELECTION OF OTHER IRS - to be sure of our choice

datex = dt.datetime(2012, 2, 20)
datey = dt.datetime(2020, 2, 19)

midprice_x = df_swaps.loc[datex, ['BID', 'ASK']].mean()
midprice_y = df_swaps.loc[datey, ['BID', 'ASK']].mean()

irs_fixed_leg_payment_dates_x = date_series(today, datex, irs_fixed_leg_freq)[1:]
irs_fixed_leg_payment_dates_y = date_series(today, datey, irs_fixed_leg_freq)[1:]

irs_mtm_x = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors, swap_type=SwapType.RECEIVER
)
irs_mtm_y = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors, swap_type=SwapType.RECEIVER
)


irs_mtm_x_10y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_10y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_10y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_x_15y_up = swap_mtm(
    midprice_x, irs_fixed_leg_payment_dates_x, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)
irs_mtm_y_15y_up = swap_mtm(
    midprice_y, irs_fixed_leg_payment_dates_y, discount_factors_15y_up, swap_type=SwapType.RECEIVER
)


irs_x_10y_dv01 = irs_mtm_x_10y_up - irs_mtm_x
irs_y_10y_dv01 = irs_mtm_y_10y_up - irs_mtm_y

irs_x_15y_dv01 = irs_mtm_x_15y_up - irs_mtm_x
irs_y_15y_dv01 = irs_mtm_y_15y_up - irs_mtm_y

A = np.array([[irs_x_10y_dv01, irs_y_10y_dv01], [irs_x_15y_dv01, irs_y_15y_dv01]])
b = np.array([-ptf_numeric_10y_dv01, -ptf_numeric_15y_dv01])

delta_hedge = la.solve(A, b)
print(f'The two positions are {round(delta_hedge[0]/min_lot)}M, {round(delta_hedge[1]/min_lot)}M')

The two positions are -4843M, 1513M
